# Entraînement des Modèles de Classification Multi-Label

In [1]:
# Importations
import sys
import os
sys.path.append('..')

from notebooks import DataLoaderExploration,Config,DataPreprocessor,FeatureEngineer,ClassifierChainsModel,BinaryRelevanceModel,\
DataVisualizer,ModelEvaluator,MultiLabelMetrics
# from src.utils.metrics import MultiLabelEvaluator
# from src.utils.visualization import ResultsVisualizer

import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json


In [2]:
# Initialisation
print(" Initialisation du notebook d'entraînement...")
def _create_spark_session():
        """Créer une session Spark optimisée"""
        return SparkSession.builder \
            .appName("models_train") \
            .config("spark.executor.memory", "4g") \
            .config("spark.driver.memory", "2g") \
            .getOrCreate()
            # .master(master) \
            # .config('spark.driver.memory', memory) \
            # .config('spark.driver.executor.memory', executor_memory) \
            # .config('spark.sql.adaptive.enabled', 'true') \
            # .config('spark.sql.adaptive.coalescePartitions.enabled', 'true') \
    
            
spark=_create_spark_session()
print(" Session Spark cree...")
# Charger la configuration
config = Config()
data_config = config.get_data_config()

# Initialiser les composants
loader = DataLoaderExploration(spark, Config)
preprocessor = DataPreprocessor(loader.spark)
evaluator_ = MultiLabelMetrics()
evaluator=ModelEvaluator(loader.spark)
visualizer = DataVisualizer()

print(" Composants initialisés")

 Initialisation du notebook d'entraînement...
 Session Spark cree...
 Composants initialisés


26/01/08 03:58:55 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


##  Chargement des données avec features

In [3]:
# Charger les données prétraitées avec features
#data_path = os.path.join(data_config['processed_path'], "features_sampled110k_arxiv_data_final.json")
#sample_path = os.path.join(data_config['processed_path'], "sampled_features_arxiv_data.json")

# # Essayer d'abord l'échantillon pour les tests rapides
# if os.path.exists(sample_path):
#     print(f" Chargement de l'échantillon: {sample_path}")
#     df = loader.load_json_data(sample_path)
# else:
#     print(f"Chargement des données complètes: {data_path}")
#     df = loader.load_json_data(data_path)
#______________________
df=loader.load_json_data("../data/processed/features_sampled110k_arxiv_data_final.json")
sampled=60000
df = df.limit(sampled)
#_______________
# Afficher les informations
print(f"Affichage Info DATASET")
loader.explore_data(df)
print(f"{'=_'*30}")

# Vérifier la présence des colonnes nécessaires
required_cols = ['categories', 'features']
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print(f"Colonnes manquantes: {missing_cols}")
    print(" Exécutez d'abord les notebooks de preprocessing et feature engineering")
else:
    print(" Toutes les colonnes nécessaires sont présentes")

Chargement des données depuis ../data/processed/features_sampled110k_arxiv_data_final.json...


26/01/08 03:59:13 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/01/08 03:59:28 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/01/08 03:59:43 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/01/08 03:59:58 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/01/08 04:00:13 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/01/08 04:00:28 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure th

 110,664 articles chargés
Affichage Info DATASET

 =Exploration basique des données ===


Nombre de lignes: 60000
Nombre de colonnes: 24

Schéma:
root
 |-- 2_grams: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- 2gram_tfidf: struct (nullable = true)
 |    |-- indices: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- size: long (nullable = true)
 |    |-- type: long (nullable = true)
 |    |-- values: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |-- abstract: string (nullable = true)
 |-- avg_word_length: double (nullable = true)
 |-- categories: string (nullable = true)
 |-- category_list: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- clean_abstract: string (nullable = true)
 |-- combined_text: string (nullable = true)
 |-- domain: string (nullable = true)
 |-- features: struct (nullable = true)
 |    |-- indices: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- size: long (nullable = true)
 |    |-- type: long

## Processing

In [4]:
df = preprocessor.run_full_preprocessing(df)

Analyse des catégories arXiv...

Top 20 des catégories les plus fréquentes:


+------------------+-----+
|category          |count|
+------------------+-----+
|cs.LG             |4937 |
|hep-ph            |3977 |
|hep-th            |3829 |
|cs.CV             |3646 |
|quant-ph          |3527 |
|cs.AI             |3171 |
|gr-qc             |2539 |
|astro-ph          |2167 |
|cond-mat.mtrl-sci |2155 |
|cs.CL             |2063 |
|cond-mat.mes-hall |2037 |
|math-ph           |1806 |
|math.MP           |1806 |
|cond-mat.str-el   |1680 |
|cond-mat.stat-mech|1607 |
|astro-ph.GA       |1586 |
|astro-ph.CO       |1556 |
|math.AP           |1468 |
|stat.ML           |1448 |
|math.CO           |1439 |
+------------------+-----+
only showing top 20 rows




 Nombre total de catégories uniques: 173

Distribution du nombre de catégories par article:


[Stage 14:==========================================>              (9 + 3) / 12]

+--------------+-----+
|num_categories|count|
+--------------+-----+
|             1|31398|
|             2|18113|
|             3| 7301|
|             4| 2350|
|             5|  691|
|             6|  127|
|             7|   16|
|             8|    3|
|             9|    1|
+--------------+-----+



In [5]:
# 2. NETTOYER LE CACHE
spark.catalog.clearCache()
print(" Cache nettoyé")

# 3. CHARGER LES DONNÉES
df = loader.load_json_data("../data/processed/processed_arxiv_data.json")
print(f" Dataset: {df.count()} articles")

# 4. FEATURE ENGINEERING (avec conversion intégrée dans engineering.py)
feature_engineer = FeatureEngineer(spark, vocab_size=5000)
df_features = feature_engineer.run_full_feature_engineering(df, text_col="combined_text")

# VÉRIFIER le type
print("\n Vérification du type:")
df_features.select("features").printSchema()
# Doit afficher: features: vector (nullable = true)

# Vérifier quelques vecteurs
print("\n Exemples de vecteurs:")
samples = df_features.select("features").take(3)
for i, row in enumerate(samples):
    vec = row['features']
    print(f"  Vecteur {i+1}: type={type(vec).__name__}, size={vec.size if hasattr(vec, 'size') else 'N/A'}")

# 5. PRÉPARER LES LABELS
br_model = BinaryRelevanceModel(spark)
df_features = br_model.prepare_labels(df_features, top_n=20)



 Sélection des 30 catégories les plus fréquentes


 30 catégories sélectionnées
 30 labels préparés
 Exemples de catégories: cs.LG, hep-ph, hep-th, cs.CV, quant-ph...

 Distribution des labels (top 10):


  label_cs_LG: 4,937 (8.2%)


  label_hep_ph: 3,977 (6.6%)


  label_hep_th: 3,829 (6.4%)


...

  label_gr_qc: 2,539 (4.2%)


  label_astro_ph: 2,167 (3.6%)


  label_cond_mat_mtrl_sci: 2,155 (3.6%)


[Stage 77:==========================================>              (9 + 3) / 12]

  label_cs_CL: 2,063 (3.4%)


##  Division des données

In [6]:
# 6. SPLIT
train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)
print(f"\n Train: {train_df.count()}")
print(f" Test: {test_df.count()}")

# 7. VÉRIFICATION DES DIMENSIONS (CRITIQUE!)
print("\n🔍 Vérification des dimensions dans train_df:")
train_samples = train_df.select("features").take(10)
train_sizes = [v['features'].size for v in train_samples]
print(f"  Tailles trouvées: {set(train_sizes)}")

if len(set(train_sizes)) > 1:
    print(" PROBLÈME: Dimensions incohérentes!")
    print(f"  Détails: {train_sizes}")
else:
    print(f" Toutes les dimensions sont identiques: {train_sizes[0]}")

26/01/08 04:02:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  Division des données (test=0.2, val=0.1)
  Division des données
 Tailles des datasets:


  Train:      43,064 échantillons


  Validation: 4,817 échantillons


  Test:       12,119 échantillons

 Taille des datasets:


  Train:      43,064 articles


  Validation: 4,817 articles


[Stage 99:>                                                         (0 + 1) / 1]

  Test:       12,119 articles


##  Approche 1: Binary Relevance

In [7]:
# ============================================
# CONVERSION FORCÉE DES VECTEURS (SOLUTION DE SECOURS)
# ============================================

from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.functions import udf
import pyspark.sql.functions as F

print(" CONVERSION FORCÉE DES VECTEURS")
print("="*60)

def convert_struct_to_vector(v):
    """
    Convertit un struct en VectorUDT de Spark ML
    """
    if v is None:
        return None
    
    try:
        # Extraire les champs du struct
        if hasattr(v, '__getitem__'):
            # C'est un Row ou dict-like
            vec_type = v['type'] if 'type' in v else v[0]
            size = v['size'] if 'size' in v else v[1]
            indices = v['indices'] if 'indices' in v else v[2]
            values = v['values'] if 'values' in v else v[3]
        else:
            # Autre format
            return None
        
        # Convertir les types (bigint → int, etc.)
        size = int(size)
        indices = [int(i) for i in indices] if indices else []
        values = [float(v) for v in values] if values else []
        
        # Créer le vecteur selon le type
        if vec_type == 0:  # Dense vector
            return Vectors.dense(values)
        else:  # Sparse vector (type == 1)
            return Vectors.sparse(size, indices, values)
            
    except Exception as e:
        print(f" Erreur conversion: {e}")
        print(f"   Type de v: {type(v)}")
        if hasattr(v, '__dict__'):
            print(f"   Contenu: {v.__dict__}")
        return None

# Créer l'UDF avec le bon type de retour
convert_vector_udf = udf(convert_struct_to_vector, VectorUDT())

# ============================================
# APPLIQUER LA CONVERSION SUR TRAIN ET TEST
# ============================================

print("\nAvant conversion:")
print("Train schema:")
train_df.select("features").printSchema()

# Convertir train_df
train_df = train_df.withColumn("features_ml", convert_vector_udf(F.col("features")))
train_df = train_df.drop("features")
train_df = train_df.withColumnRenamed("features_ml", "features")

# Convertir test_df
test_df = test_df.withColumn("features_ml", convert_vector_udf(F.col("features")))
test_df = test_df.drop("features")
test_df = test_df.withColumnRenamed("features_ml", "features")

print("\n Après conversion:")
print("Train schema:")
train_df.select("features").printSchema()

# Vérifier qu'on a bien "vector" maintenant
schema_type = str(train_df.schema["features"].dataType)
if "vector" in schema_type.lower():
    print("\nSUCCÈS ! Type correct: VectorUDT")
else:
    print(f"\n ÉCHEC ! Type actuel: {schema_type}")

# Afficher un exemple pour vérifier
print("\n Exemple de vecteur:")
sample = train_df.select("features").take(1)[0]
print(f"   Type Python: {type(sample['features'])}")
print(f"   Valeur: {sample['features']}")

# IMPORTANT: Recacher les DataFrames après conversion
train_df = train_df.cache()
test_df = test_df.cache()

print("\nConversion terminée et DataFrames cachés")
print("="*60)




train_df.printSchema()


 CONVERSION FORCÉE DES VECTEURS

Avant conversion:
Train schema:
root
 |-- features: struct (nullable = true)
 |    |-- indices: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- size: long (nullable = true)
 |    |-- type: long (nullable = true)
 |    |-- values: array (nullable = true)
 |    |    |-- element: double (containsNull = true)


 Après conversion:
Train schema:
root
 |-- features: vector (nullable = true)


SUCCÈS ! Type correct: VectorUDT

 Exemple de vecteur:


[Stage 103:>                                                        (0 + 1) / 1]

   Type Python: <class 'pyspark.ml.linalg.DenseVector'>
   Valeur: [5.725384940543265,7.856177861817514,2.8522429453145564,2.408533139457646,4.5333950021853635,2.137719486125306,2.340478354343428,1.8264668208362091,3.391172347714612,11.87661676214998,8.63569926020445,8.19783665539142,3.509861590954533,7.503531403291201,6.5543075386273415,4.211201807786055,4.279281020004332,1.5741487488081922,2.4371691206935986,3.2008758761854974,1.3599424338849182,3.010341703950084,19.01955107575928,3.4389967947640847,3.197332129398302,2.892823593250162,4.0003845121710295,34.44576081069564,1.3584649130363733,3.2432522176379894,13.880657742150486,455.0,85.0,5.352941176470588,0.47058823529411764]

Conversion terminée et DataFrames cachés
root
 |-- 2_grams: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- 2gram_tfidf: struct (nullable = true)
 |    |-- indices: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- size: long (nullable = true)
 |  

In [9]:
# ============================================
# NETTOYAGE COMPLET
# ============================================

print("🧹 NETTOYAGE COMPLET")
print("="*60)

# 1. Unpersist tous les DataFrames
try:
    if 'train_df' in locals():
        train_df.unpersist()
    if 'test_df' in locals():
        test_df.unpersist()
    if 'df_features' in locals():
        df_features.unpersist()
    print("✅ Cache nettoyé")
except:
    pass

# 2. Nettoyer le cache Spark
spark.catalog.clearCache()

print("✅ Nettoyage terminé")
print("="*60)


🧹 NETTOYAGE COMPLET
✅ Cache nettoyé
✅ Nettoyage terminé


In [10]:
print(" APPROCHE 1: BINARY RELEVANCE")

# Entraîner le modèle
print("\n Début de l'entraînement...")
br_model.train(train_df, features_col="features")

# Prédictions sur le test set
print("\n Génération des prédictions...")
predictions_br = br_model.predict(test_df)

# Évaluation
print("\nÉvaluation du modèle...")
br_pred_cols = [f"{col}_pred" for col in label_columns]
metrics_br = evaluator.evaluate_model(
    predictions_br,
    label_columns,
    br_pred_cols,
    model_name="Binary Relevance"
)

# Sauvegarder le modèle
models_path = data_config.get('models_path', 'data/models')
br_output_dir = os.path.join(models_path, "binary_relevance")
br_model.save_models(br_output_dir)
print(f"\n Modèle sauvegardé: {br_output_dir}")

 APPROCHE 1: BINARY RELEVANCE

 Début de l'entraînement...
Entraînement Binary Relevance
  Entraînement pour label_cs_LG...
 Correction du type de vecteur...
 Type de vecteur corrigé ✓


26/01/08 04:12:06 WARN TaskSetManager: Lost task 0.0 in stage 109.0 (TID 452) (10.188.160.181 executor 1): java.lang.IllegalArgumentException: requirement failed: Dimensions mismatch when adding new sample. Expecting 35 but got 48.
	at scala.Predef$.require(Predef.scala:281)
	at org.apache.spark.ml.stat.SummarizerBuffer.add(Summarizer.scala:495)
	at org.apache.spark.ml.stat.SummarizerBuffer.add(Summarizer.scala:552)
	at org.apache.spark.ml.stat.Summarizer$.$anonfun$getClassificationSummarizers$1(Summarizer.scala:235)
	at scala.collection.TraversableOnce$folder$1.apply(TraversableOnce.scala:196)
	at scala.collection.TraversableOnce$folder$1.apply(TraversableOnce.scala:194)
	at scala.collection.Iterator.foreach(Iterator.scala:943)
	at scala.collection.Iterator.foreach$(Iterator.scala:943)
	at scala.collection.AbstractIterator.foreach(Iterator.scala:1431)
	at scala.collection.TraversableOnce.foldLeft(TraversableOnce.scala:199)
	at scala.collection.TraversableOnce.foldLeft$(TraversableOnce

Py4JJavaError: An error occurred while calling o486.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 109.0 failed 4 times, most recent failure: Lost task 0.3 in stage 109.0 (TID 455) (10.188.160.181 executor 1): java.lang.IllegalArgumentException: requirement failed: Dimensions mismatch when adding new sample. Expecting 35 but got 48.
	at scala.Predef$.require(Predef.scala:281)
	at org.apache.spark.ml.stat.SummarizerBuffer.add(Summarizer.scala:495)
	at org.apache.spark.ml.stat.SummarizerBuffer.add(Summarizer.scala:552)
	at org.apache.spark.ml.stat.Summarizer$.$anonfun$getClassificationSummarizers$1(Summarizer.scala:235)
	at scala.collection.TraversableOnce$folder$1.apply(TraversableOnce.scala:196)
	at scala.collection.TraversableOnce$folder$1.apply(TraversableOnce.scala:194)
	at scala.collection.Iterator.foreach(Iterator.scala:943)
	at scala.collection.Iterator.foreach$(Iterator.scala:943)
	at scala.collection.AbstractIterator.foreach(Iterator.s

##  Approche 2: Classifier Chains

In [ ]:
print("\n" + "="*30)
print("APPROCHE 2: CLASSIFIER CHAINS")


# Préparer les données pour Classifier Chains
print("\n Préparation des données pour Classifier Chains...")

# Créer les mêmes colonnes de labels dans les DataFrames d'entraînement et de test
train_df_cc = train_df
test_df_cc = test_df

for col in label_columns:
    # Si la colonne n'existe pas, la créer
    if col not in train_df_cc.columns:
        # Extraire le nom de la catégorie du nom de colonne
        cat_name = col.replace("label_", "").replace("_", ".")
        train_df_cc = train_df_cc.withColumn(
            col,
            F.when(F.array_contains(F.split(F.col("categories"), " "), cat_name), 1.0).otherwise(0.0)
        )
        test_df_cc = test_df_cc.withColumn(
            col,
            F.when(F.array_contains(F.split(F.col("categories"), " "), cat_name), 1.0).otherwise(0.0)
        )

# Entraîner le modèle
print("\n Début de l'entraînement...")
cc_model = ClassifierChainsModel(loader.spark)
cc_model.train(train_df_cc, label_columns, features_col="features")

# Prédictions
print("\n Génération des prédictions...")
predictions_cc = cc_model.predict(test_df_cc)

# Évaluation
print("\n Évaluation du modèle...")
cc_pred_cols = [f"{col}_pred" for col in label_columns]
metrics_cc = evaluator.evaluate_model(
    predictions_cc,
    label_columns,
    cc_pred_cols,
    model_name="Classifier Chains"
)

# Sauvegarder le modèle
cc_output_dir = os.path.join(models_path, "classifier_chains")
cc_model.save_models(cc_output_dir)
print(f"\n Modèle sauvegardé: {cc_output_dir}")

##  Comparaison des modèles

In [ ]:
print(" COMPARAISON DES MODÈLES")

# Collecter les métriques pour comparaison
all_metrics = [metrics_br, metrics_cc]
best_model = evaluator.compare_models(all_metrics)

# Sauvegarder les résultats
comparison_results = {
    "models_trained": ["Binary Relevance", "Classifier Chains"],
    "metrics": all_metrics,
    "best_model": best_model["model"],
    "top_categories": top_categories,
    "timestamp": str(pd.Timestamp.now())
}

# Créer le dossier de rapports
reports_dir = "reports"
os.makedirs(reports_dir, exist_ok=True)

# Sauvegarder en JSON
results_file = os.path.join(reports_dir, "model_comparison.json")
with open(results_file, 'w') as f:
    json.dump(comparison_results, f, indent=4, default=str)

print(f"\n Résultats sauvegardés: {results_file}")

## Visualisation des résultats

In [ ]:
# Créer des visualisations
print(" Création des visualisations...")

# 1. Bar chart comparatif
models = ["Binary Relevance", "Classifier Chains"]
metrics_to_plot = ["hamming_loss", "subset_accuracy", "micro_f1", "macro_f1"]

plt.figure(figsize=(12, 6))

for i, metric in enumerate(metrics_to_plot):
    plt.subplot(2, 2, i + 1)
    
    values = [m[metric] for m in all_metrics]
    colors = ['lightblue', 'lightgreen']
    
    bars = plt.bar(models, values, color=colors, alpha=0.7)
    
    plt.title(metric.replace('_', ' ').title())
    plt.ylabel('Score')
    plt.ylim(0, 1 if metric != 'hamming_loss' else 0.5)
    plt.grid(axis='y', alpha=0.3)
    
    # Ajouter les valeurs
    for bar, value in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{value:.3f}', ha='center', va='bottom')

plt.suptitle('Comparaison des Modèles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(reports_dir, "model_comparison_chart.png"), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2. Heatmap des prédictions (exemple avec Binary Relevance)
print("\n🔥 Heatmap des prédictions (Binary Relevance - échantillon)")

# Prendre un échantillon pour la visualisation
sample_predictions = predictions_br.select(
    *[col for col in predictions_br.columns if col.endswith('_pred')]
).limit(100).toPandas()

# Calculer la matrice de corrélation
correlation_matrix = sample_predictions.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, 
            square=True, linewidths=.5, cbar_kws={"shrink": .8})
plt.title('Corrélation entre les Prédictions de Labels (Binary Relevance)')
plt.tight_layout()
plt.savefig(os.path.join(reports_dir, "predictions_correlation.png"), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3. Analyse des erreurs
print("\n🔍 Analyse des erreurs par catégorie")

# Calculer la précision par catégorie pour Binary Relevance
category_accuracy = []

for i, label_col in enumerate(label_columns):
    pred_col = f"{label_col}_pred"
    
    # Calculer l'accuracy pour cette catégorie
    correct = predictions_br.filter(F.col(label_col) == F.col(pred_col)).count()
    total = predictions_br.count()
    accuracy = correct / total if total > 0 else 0
    
    category_accuracy.append({
        'category': label_col,
        'accuracy': accuracy,
        'correct': correct,
        'total': total
    })

# Trier par accuracy
category_accuracy.sort(key=lambda x: x['accuracy'])

print("\n📊 Catégories les plus difficiles (précision la plus basse):")
for i, cat in enumerate(category_accuracy[:10]):
    print(f"  {i+1}. {cat['category']}: {cat['accuracy']:.3f} ({cat['correct']}/{cat['total']})")

print("\n🏆 Catégories les plus faciles (précision la plus haute):")
for i, cat in enumerate(category_accuracy[-10:]):
    print(f"  {i+1}. {cat['category']}: {cat['accuracy']:.3f} ({cat['correct']}/{cat['total']})")

## 💾 Export des résultats

In [ ]:
print("\n💾 Export des résultats finaux...")

# 1. Exporter les métriques en CSV
import pandas as pd

# Créer un DataFrame avec les métriques
metrics_df = pd.DataFrame(all_metrics)
metrics_csv = os.path.join(reports_dir, "model_metrics.csv")
metrics_df.to_csv(metrics_csv, index=False)
print(f"📄 Métriques exportées: {metrics_csv}")

# 2. Exporter les catégories utilisées
categories_df = pd.DataFrame({
    'category': top_categories,
    'label_column': label_columns
})
categories_csv = os.path.join(reports_dir, "categories_used.csv")
categories_df.to_csv(categories_csv, index=False)
print(f"📄 Catégories exportées: {categories_csv}")

# 3. Exporter un échantillon de prédictions
sample_output = predictions_br.select(
    "id",
    *label_columns[:5],
    *[f"{col}_pred" for col in label_columns[:5]]
).limit(50)

sample_df = sample_output.toPandas()
sample_csv = os.path.join(reports_dir, "predictions_sample.csv")
sample_df.to_csv(sample_csv, index=False)
print(f"📄 Échantillon de prédictions: {sample_csv}")

## 🧹 Nettoyage

In [ ]:
# Nettoyer la mémoire
print("\n🧹 Nettoyage de la mémoire...")

train_df.unpersist()
test_df.unpersist()

# Arrêter la session Spark
loader.spark.stop()

print("\n" + "="*60)
print("✅ NOTEBOOK TERMINÉ AVEC SUCCÈS!")
print("="*60)
print(f"\n📊 RÉSULTATS PRINCIPAUX:")
print(f"  • Meilleur modèle: {best_model['model']}")
print(f"  • Micro F1: {best_model['micro_f1']:.4f}")
print(f"  • Hamming Loss: {best_model['hamming_loss']:.4f}")
print(f"  • Modèles sauvegardés dans: {models_path}")
print(f"  • Rapports générés dans: {reports_dir}")